# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Damasoumana1/flyrank-ml-internship-july2026/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

# Pass the HF token explicitly to DuckDB's HTTP filesystem
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute(f"SET http_retries = 3")
con.execute(f"SET http_timeout = 60")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MARCH = (
    f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
)

FACT_APRIL = (
    f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
)

ANALYSIS_MONTH = "2026-03"
OUTCOME_MONTH = "2026-04"

print({
    "analysis_month": ANALYSIS_MONTH,
    "outcome_month": OUTCOME_MONTH
})

{'analysis_month': '2026-03', 'outcome_month': '2026-04'}


In [4]:
# # Vérifier que le secret HF est bien disponible
# from google.colab import userdata

# token = userdata.get("HF_TOKEN")

# print("Token found:", token is not None)
# print("Token prefix:", token[:3] if token else None)
# print("Token length:", len(token) if token else 0)

Token found: True
Token prefix: hf_
Token length: 37


In [5]:
# from huggingface_hub import whoami

# info = whoami(token=token)
# print("Authenticated as:", info["name"])

Authenticated as: Dama12


Dataset access

In [14]:
test = con.sql(f"""
    SELECT *
    FROM {FACT_MARCH}
    LIMIT 5
""").df()

display(test)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

**Contract answer.** One row in the feature frame represents one pseudonymized content item for one pseudonymized client (`client_hash_id`, `content_hash_id`). The decision-time feature window is March 2026, aggregated from the daily fact table. The observed outcome window is April 2026: a content item is marked as declining when April impressions are below 80% of March impressions. This is an observed directional proxy for prioritization, not a causal claim that a refresh would recover the page.

The daily source itself has a finer grain: one row represents one `report_date × client_hash_id × content_hash_id`. Query 1 checks that documented grain before the monthly aggregation.


In [15]:
# Verification query 1 — daily grain.
# An empty result is the expected evidence that the documented grain holds.
GRAIN_SQL = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM {FACT_MARCH}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
ORDER BY duplicate_rows DESC
LIMIT 10
"""

grain_check = con.sql(GRAIN_SQL).df()
print("Duplicate daily-grain groups returned:", len(grain_check))
display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate daily-grain groups returned: 0


,report_date,client_hash_id,content_hash_id,duplicate_rows


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

**Features — knowable at the decision moment.**

- `march_impressions`: observed search exposure accumulated during March.
- `march_clicks`: observed search clicks accumulated during March.
- `march_avg_position`: average observed GSC position during March; zero or missing values are not interpreted as a real rank of zero.
- `march_days_with_impressions`: number of March days with at least one impression.
- `march_ga4_sessions`: March GA4 sessions counted only when `ga4_data_available IS TRUE`; otherwise the value is treated as unavailable for this feature frame.

**Label / proxy.** `is_declining_next30` is computed from April impressions compared with March impressions. It is the observed outcome used for this exercise and must never be used as a feature.

**Context.** `client_hash_id`, `content_hash_id`, `report_date`, `ANALYSIS_MONTH`, and `OUTCOME_MONTH` are for grouping, joining, time alignment, or auditability. The hashed IDs are not model features.

**Excluded.** The `_sample` table and June 2026 are excluded from label development because the repository warns that `_sample` is the final month. Any product-decision fields such as `health_score`, `priority_score`, or `action_type` are excluded because they would reproduce an existing decision rather than discover a signal. `trend_direction` and `trend_pct` are excluded when they are used to derive a label, because that would be direct label leakage. Raw client names, domains, URLs, titles, queries, and tokens are excluded for privacy.


In [16]:
# This cell contains field definitions and safety checks, not a fourth verification query.
FEATURE_COLS = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_days_with_impressions",
    "march_ga4_sessions",
]
LABEL_COL = "is_declining_next30"
CONTEXT_COLS = ["client_hash_id", "content_hash_id", "ANALYSIS_MONTH", "OUTCOME_MONTH"]
EXCLUDED_COLS = [
    "trend_direction", "trend_pct", "health_score", "priority_score",
    "action_type", "client_name", "domain", "url", "title", "query", "HF_TOKEN",
]

assert len(FEATURE_COLS) == 5
assert LABEL_COL not in FEATURE_COLS
assert not set(FEATURE_COLS).intersection(EXCLUDED_COLS)
print("Features:", FEATURE_COLS)
print("Label:", LABEL_COL)
print("Context only:", CONTEXT_COLS)
print("Excluded categories checked:", len(EXCLUDED_COLS))


Features: ['march_impressions', 'march_clicks', 'march_avg_position', 'march_days_with_impressions', 'march_ga4_sessions']
Label: is_declining_next30
Context only: ['client_hash_id', 'content_hash_id', 'ANALYSIS_MONTH', 'OUTCOME_MONTH']
Excluded categories checked: 11


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 3 checks availability using `IS TRUE`, as required by the warehouse contract. A NULL or FALSE availability flag does not count as available GA4 data. The same query also builds the bounded feature frame used for the deliberate leakage experiment. The `LIMIT` keeps development RAM bounded; remove it only for a final, explicitly monitored pass.


In [ ]:
# Verification query 3 — availability with IS TRUE + five-feature frame.
# The outcome is future-looking: March features -> April observed decline proxy.
FEATURES_SQL = f"""
WITH march_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position,
        SUM(CASE WHEN COALESCE(gsc_impressions, 0) > 0 THEN 1 ELSE 0 END) AS march_days_with_impressions,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN COALESCE(ga4_sessions, 0) ELSE 0 END) AS march_ga4_sessions
    FROM {FACT_MARCH}
    GROUP BY 1, 2
),
april_outcomes AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS april_impressions
    FROM {FACT_APRIL}
    GROUP BY 1, 2
),
availability AS (
    SELECT
        COUNT(*) AS total_march_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_true_rows
    FROM {FACT_MARCH}
),
joined AS (
    SELECT
        m.*,
        a.april_impressions,
        CAST(a.april_impressions < 0.8 * m.march_impressions AS INTEGER) AS is_declining_next30
    FROM march_features m
    INNER JOIN april_outcomes a
        USING (client_hash_id, content_hash_id)
    WHERE m.march_impressions > 0
)
SELECT
    j.*,
    av.total_march_rows,
    av.ga4_true_rows,
    COUNT(*) OVER () AS feature_frame_rows
FROM joined j
CROSS JOIN availability av
LIMIT 100000
"""

feature_frame = con.sql(FEATURES_SQL).df()
print("Feature frame rows returned:", len(feature_frame))
print("Availability rows where ga4_data_available IS TRUE:", int(feature_frame["ga4_true_rows"].iloc[0]))
display(feature_frame.head())
display(feature_frame[["total_march_rows", "ga4_true_rows", "feature_frame_rows"]].head(1))


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This slice cannot tell us that refreshing a page *caused* recovery. It can only provide observed search and analytics outcomes and a directional prioritization proxy. The warehouse is an unbalanced panel: clients have different history depths, so a single calendar month is not equally informative for every client. Early rows may be GSC-only; GA4 zeroes or missing values must not be interpreted as “no engagement” unless `ga4_data_available IS TRUE` was checked. The March feature window and April outcome window are deliberately kept separate, but the label still depends on an 80% threshold chosen for this exercise rather than a validated business capacity.

**Named limitation:** the contract uses a bounded development frame of up to 100,000 content items and therefore the printed leakage/model numbers are a smoke test, not a population estimate for the full warehouse.


In [ ]:
# Deliberate leakage experiment — performed after the contract queries.
# The label-derived feature is added on purpose, its score is measured, then it is deleted.
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

model_df = feature_frame.dropna(subset=FEATURE_COLS + [LABEL_COL, "client_hash_id"]).copy()
X = model_df[FEATURE_COLS].copy()
y = model_df[LABEL_COL].astype(int)
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

# Trap: use the future-derived label itself as an input.
X_leaky = X.copy()
X_leaky["label_derived_leak"] = y.to_numpy()
leaky_model = DecisionTreeClassifier(max_depth=2, random_state=42)
leaky_model.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
leaky_pred = leaky_model.predict(X_leaky.iloc[test_idx])
leaky_accuracy = accuracy_score(y.iloc[test_idx], leaky_pred)
print(f"Leaky accuracy (label-derived column intentionally included): {leaky_accuracy:.3f}")

# Delete the trap before keeping the honest result.
del X_leaky["label_derived_leak"]
honest_model = DecisionTreeClassifier(max_depth=4, random_state=42)
honest_model.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
honest_pred = honest_model.predict(X_leaky.iloc[test_idx])
honest_accuracy = accuracy_score(y.iloc[test_idx], honest_pred)
print(f"Honest accuracy after deleting the leak: {honest_accuracy:.3f}")

LEAKAGE_REMOVED = "label_derived_leak" not in X_leaky.columns
assert LEAKAGE_REMOVED
print("Leakage column removed:", LEAKAGE_REMOVED)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.